In [3]:
import torch
import torch.nn as nn
from transformers import VisionEncoderDecoderModel, ViTForImageClassification
input_tensor = torch.randn(1, 3, 224, 224)  # Example input tensor

# clip-vit

In [ ]:
from transformers import VisionEncoderDecoderModel, ViTModel, ViTForImageClassification, CLIPModel
from transformers import CLIPVisionModel, CLIPImageProcessor
model = CLIPModel.from_pretrained('openai/clip-vit-large-patch14')

In [20]:
model

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05,

In [ ]:
class WrappedModel(torch.nn.Module):
    def __init__(self, model, type_of_output='cls',normalization=False):
        super().__init__()
        self.model = model
        self.type_of_output = type_of_output
        self.normalization = normalization

    def forward(self, x):
        image_features = self.model.get_image_features(x)
        # Normalize the features (optional but common)
        if self.normalization:
            image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
        return image_features

class CLIPViTClsExtractor(nn.Module):
    """
    Works with either:
      - transformers.CLIPVisionModel
      - transformers.CLIPModel  (uses .vision_model internally)
    Returns [CLS] from the specified vision block.
    """
    def __init__(self, model, layer_index: int = 12):
        super().__init__()
        # If it's a full CLIPModel, grab the vision tower
        self.vision = getattr(model, "vision_model", model)
        num_layers = self.vision.config.num_hidden_layers
        assert 1 <= layer_index <= num_layers, f"layer_index must be in [1, {num_layers}]"
        self.layer_index = layer_index

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        # Run ONLY the vision encoder; request hidden states
        out = self.vision(pixel_values=pixel_values, output_hidden_states=True)
        hs = out.hidden_states[self.layer_index]   # [B, seq_len, hidden_dim]
        cls = hs[:, 0, :]                          # [CLS]
        return cls


In [31]:
input_tensor = torch.randn(1, 3, 224, 224)  # Example input tensor
wrapped_model = WrappedModel(model, type_of_output='cls', normalization=False)
wrapped_12_model = CLIPViTClsExtractor(model, layer_index=12)

In [33]:
out=wrapped_model(input_tensor)
out.shape

torch.Size([1, 768])

In [32]:
out=wrapped_12_model(input_tensor)
out.shape

torch.Size([1, 1024])

In [9]:
wrapped_model

WrappedModel(
  (model): CLIPModel(
    (text_model): CLIPTextTransformer(
      (embeddings): CLIPTextEmbeddings(
        (token_embedding): Embedding(49408, 768)
        (position_embedding): Embedding(77, 768)
      )
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-11): 12 x CLIPEncoderLayer(
            (self_attn): CLIPAttention(
              (k_proj): Linear(in_features=768, out_features=768, bias=True)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (mlp): CLIPMLP(
              (activation_fn): QuickGELUActivation()
              (fc1): Linear(in_features=768, out_features=3072, bias=True)
              (fc2): Linear(in_features=3072, out_features=768, bias=True)
  

# beit 

In [37]:
from transformers import BeitForImageClassification
model = BeitForImageClassification.from_pretrained('microsoft/beit-large-patch16-384',output_hidden_states=True)

In [38]:
model

BeitForImageClassification(
  (beit): BeitModel(
    (embeddings): BeitEmbeddings(
      (patch_embeddings): BeitPatchEmbeddings(
        (projection): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): BeitEncoder(
      (layer): ModuleList(
        (0): BeitLayer(
          (attention): BeitAttention(
            (attention): BeitSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=False)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
              (relative_position_bias): BeitRelativePositionBias()
            )
            (output): BeitSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
 

In [39]:
class BEiTClsExtractor(nn.Module):
    """
    Extract the [CLS] token from the N-th BEiT encoder layer (default: 12th).
    Works with either:
      - transformers.BeitModel
      - transformers.BeitForImageClassification (uses .beit internally)
    Assumes pixel_values are already preprocessed (resize + normalize).
    """
    def __init__(self, beit_or_classifier_model, layer_index: int = 12):
        super().__init__()
        # If it's a classifier, reach the base vision model via .beit
        self.beit = getattr(beit_or_classifier_model, "beit", beit_or_classifier_model)
        #num_layers = self.beit.config.num_hidden_layers
        #assert 1 <= layer_index <= num_layers, f"layer_index must be in [1, {num_layers}]"
        self.layer_index = layer_index

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        # Request hidden states so we can index the chosen layer
        out = self.beit(pixel_values=pixel_values, output_hidden_states=True)
        # hidden_states[0] = embeddings (patch+pos); hidden_states[k] = after k-th block
        hs = out.hidden_states[self.layer_index]     # [B, seq_len, hidden_dim]
        cls = hs[:, 0, :]                            # [CLS] token
        return cls

In [40]:
model_inter=BEiTClsExtractor(model, layer_index=12)

In [41]:
out=model_inter(input_tensor)

In [44]:
out.shape

torch.Size([1, 1024])

# deit

In [4]:
from transformers import DeiTForImageClassificationWithTeacher
model = ViTForImageClassification.from_pretrained(f'facebook/deit-tiny-patch16-224',output_hidden_states=True)

In [5]:
model

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=192, out_features=192, bias=True)
              (key): Linear(in_features=192, out_features=192, bias=True)
              (value): Linear(in_features=192, out_features=192, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=192, out_features=192, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=192, out_features=768, bias=True)
            (intermedi

In [6]:

class DeiTClsExtractor(nn.Module):
    """
    Extract the [CLS] token from the N-th ViT/DeiT encoder layer (default: 12th).
    Works with:
      - transformers.ViTForImageClassification  (DeiT often uses this class)
      - transformers.ViTModel
      - transformers.DeiTModel (if you use distilled variants; see token_index)
    Assumes pixel_values are already resized & normalized.
    """
    def __init__(self, vit_or_classifier, layer_index: int = 6, token_index: int = 0):
        super().__init__()
        # If it's a classifier, grab the base ViT
        self.vit = getattr(vit_or_classifier, "vit", vit_or_classifier)
        self.layer_index = layer_index

        # Which token to read: 0 = CLS; for DeiT distilled models, 1 can be distillation token
        self.token_index = token_index

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        out = self.vit(pixel_values=pixel_values, output_hidden_states=True)
        # hidden_states[0]: embeddings (after patch+pos(+cls/distill))
        # hidden_states[k]: output after k-th transformer block
        hs = out.hidden_states[self.layer_index]        # [B, seq_len, hidden_dim]
        cls_like = hs[:, self.token_index, :]           # pick CLS (or distill) token
        return cls_like

wrapped = DeiTClsExtractor(model, layer_index=6)


In [7]:
out=wrapped(input_tensor)
out.shape

torch.Size([1, 192])

# trocr

In [50]:
name='trocr-large-stage1'
model = VisionEncoderDecoderModel.from_pretrained(f'microsoft/{name}')

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-large-stage1 and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [47]:
model

VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-23): 24 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=False)
              (key): Linear(in_features=1024, out_features=1024, bias=False)
              (value): Linear(in_features=1024, out_features=1024, bias=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=1024, out_features=4096, bias=True)
    

In [51]:
class TrOCRViTClsExtractor(nn.Module):
    """
    Extract the [CLS] token from the N-th ViT encoder layer (default: 12th).
    Works with:
      - transformers.VisionEncoderDecoderModel (uses .encoder)
      - transformers.ViTModel
    Assumes pixel_values are already preprocessed (resize + normalize).
    """
    def __init__(self, trocr_or_vit, layer_index: int = 12):
        super().__init__()
        # If it's a VisionEncoderDecoderModel, grab the vision encoder
        self.encoder = getattr(trocr_or_vit, "encoder", trocr_or_vit)
        #num_layers = self.encoder.config.num_hidden_layers
        #assert 1 <= layer_index <= num_layers, f"layer_index must be in [1, {num_layers}]"
        self.layer_index = layer_index

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        # Run ONLY the vision encoder and request hidden states
        out = self.encoder(pixel_values=pixel_values, output_hidden_states=True)
        # hidden_states[0] = embeddings (patch+pos+cls); hidden_states[k] = after k-th block
        hs = out.hidden_states[self.layer_index]    # [B, seq_len, hidden_dim]
        cls = hs[:, 0, :]                           # [CLS]
        return cls

In [52]:
input_tensor = torch.randn(1, 3, 384, 384)  # Example input tensor
model=TrOCRViTClsExtractor(model, layer_index=12)
out=model(input_tensor)
out.shape

torch.Size([1, 1024])